# 06 - Predicción URL Top25 con scaler

Este notebook usa `feature_extractor_top25.py`, el mismo extractor que usa la API. No define funciones duplicadas de extracci?n.

In [1]:
import os
import sys
from pathlib import Path

sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

BASE_DIR = Path.cwd()
if not (BASE_DIR / "feature_extractor_top25.py").exists():
    BASE_DIR = Path(r"C:\phishing-url-classifier\phishing-url-ai")

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print("BASE_DIR:", BASE_DIR)


BASE_DIR: c:\phishing-url-classifier\phishing-url-ai


In [2]:
import importlib
import pandas as pd
import sklearn

import feature_extractor_top25 as extractor
extractor = importlib.reload(extractor)

extract_top25_features = extractor.extract_top25_features
scale_top25_features = extractor.scale_top25_features
load_feature_names = extractor.load_feature_names
load_continuous_features = extractor.load_continuous_features
load_model = extractor.load_model


def predict_from_raw_features(url, raw_features):
    scaled_features = scale_top25_features(raw_features)
    prediction = int(model.predict(scaled_features)[0])

    probability = None
    if hasattr(model, "predict_proba"):
        classes = list(model.classes_)
        probabilities = model.predict_proba(scaled_features)[0]
        probability = float(probabilities[classes.index(1)]) if 1 in classes else float(max(probabilities))

    return {
        "url": extractor.normalize_url(url),
        "prediction": prediction,
        "label_extractor": "phishing" if prediction == 1 else "legitimate",
        "label_api": "phishing" if prediction == 1 else "no-phishing",
        "probability": probability,
        "raw_features": raw_features,
        "scaled_features": scaled_features,
    }

MODEL_FEATURES = load_feature_names()
CONTINUOUS_FEATURES = load_continuous_features()
model = load_model()

print("Extractor:", extractor.__file__)
print("scikit-learn:", sklearn.__version__)
print("Cantidad de features:", len(MODEL_FEATURES))
print("Cantidad de features continuas:", len(CONTINUOUS_FEATURES))
print("Orden coincide:", list(model.feature_names_in_) == MODEL_FEATURES)
print("Clases:", list(model.classes_))


Extractor: c:\phishing-url-classifier\phishing-url-ai\feature_extractor_top25.py
scikit-learn: 1.7.2
Cantidad de features: 25
Cantidad de features continuas: 20
Orden coincide: True
Clases: [0, 1]


## Predicción individual

Cambia `URL_TEST` si quieres probar otra URL. `USE_NETWORK_FEATURES = True` replica el flujo completo de la API.

In [3]:
URL_TEST = "https://www.google.com"
USE_NETWORK_FEATURES = True

# Importante: se extraen features una sola vez, igual que en la API.
raw_features = extract_top25_features(
    URL_TEST,
    use_network_features=USE_NETWORK_FEATURES,
)
result = predict_from_raw_features(URL_TEST, raw_features)
scaled_features = result["scaled_features"]

print("URL:", result["url"])
print("Prediction:", result["prediction"])
print("Label extractor:", result["label_extractor"])
print("Label API:", result["label_api"])
print("Probability phishing:", result["probability"])
print("Raw shape:", raw_features.shape)
print("Scaled shape:", scaled_features.shape)
print("Orden raw correcto:", list(raw_features.columns) == MODEL_FEATURES)
print("Orden scaled correcto:", list(scaled_features.columns) == MODEL_FEATURES)
print("Nulos raw/scaled:", int(raw_features.isna().sum().sum()), int(scaled_features.isna().sum().sum()))


URL: https://www.google.com
Prediction: 0
Label extractor: legitimate
Label API: no-phishing
Probability phishing: 0.33
Raw shape: (1, 25)
Scaled shape: (1, 25)
Orden raw correcto: True
Orden scaled correcto: True
Nulos raw/scaled: 0 0


## Features sin escalar

In [4]:
display(raw_features.T.rename(columns={0: "valor_sin_escalar"}))


,valor_sin_escalar
google_index,0.000000
domain_age,10525.000000
nb_hyperlinks,11.000000
nb_www,1.000000
phish_hints,0.000000
safe_anchor,0.000000
longest_word_path,0.000000
ratio_digits_url,0.000000
ratio_intHyperlinks,0.909091
longest_words_raw,6.000000


## Features escaladas

In [5]:
display(scaled_features.T.rename(columns={0: "valor_escalado"}))


,valor_escalado
google_index,0.000000
domain_age,2.079508
nb_hyperlinks,-0.456925
nb_www,1.000000
phish_hints,-0.388992
safe_anchor,-0.948603
longest_word_path,-0.457664
ratio_digits_url,-0.594606
ratio_intHyperlinks,0.814434
longest_words_raw,-0.425404


## Prueba con varias URLs

In [6]:
urls = [
    "https://www.google.com",
    "https://www.pcfactory.cl",
    "http://login-banco-seguro.xyz",
]

rows = []
local_results = {}

for url in urls:
    # Una sola extracci?n por URL.
    raw = extract_top25_features(url, use_network_features=USE_NETWORK_FEATURES)
    item = predict_from_raw_features(url, raw)
    local_results[url] = item
    rows.append({
        "url": item["url"],
        "label_extractor": item["label_extractor"],
        "label_api": item["label_api"],
        "prediction": item["prediction"],
        "probability_phishing": item["probability"],
    })

pd.DataFrame(rows)


,url,label_extractor,label_api,prediction,probability_phishing
0,https://www.google.com,legitimate,no-phishing,0,0.330000
1,https://www.pcfactory.cl,legitimate,no-phishing,0,0.160000
2,http://login-banco-seguro.xyz,phishing,phishing,1,0.796667
